In [1]:
# Install dependencies
!pip install datasets transformers torchvision torch
!pip install -q uv ruff mypy pyright

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet50, ResNet50_Weights
from datasets import load_dataset
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import numpy as np
import copy
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 146.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.3/230.3 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.7 MB/s eta 0:00:00
Using device: cuda


In [9]:
from torchvision.datasets import Imagenette
from torch.utils.data import DataLoader, Subset

# Define standard ResNet50 transforms
weights = ResNet50_Weights.IMAGENET1K_V2
preprocess = weights.transforms()

print("Downloading and loading Imagenette via torchvision...")

# Load Imagenette natively via torchvision (Bypasses Hugging Face datasets)
# size="160px" keeps the download and processing fast for Colab
train_dataset = Imagenette(
    root="./data",
    split="train",
    size="160px",
    download=True,
    transform=preprocess
)

val_dataset = Imagenette(
    root="./data",
    split="val",
    size="160px",
    download=True,
    transform=preprocess
)

# Use a smaller subset for demonstration purposes to avoid timeouts in Colab
demo_indices = list(range(2000))
train_subset = Subset(train_dataset, demo_indices)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Train subset size: {len(train_subset)}")
print(f"Validation set size: {len(val_dataset)}")

100%|██████████| 99.0M/99.0M [00:09<00:00, 10.4MB/s]


Train subset size: 2000
Validation set size: 3925


In [10]:
def get_model(frozen=True):
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

    if frozen:
        # Linear Probe: Freeze all layers except the final classification head
        for param in model.parameters():
            param.requires_grad = False

    # Modify the final layer for 10 classes (Imagenette)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 10)

    return model.to(device)

In [12]:
def compute_vog(model, dataloader, epochs=3):
    """
    Computes Variance of Gradients (VoG) over K early training epochs.
    Returns an array of VoG scores corresponding to the dataset indices.
    """
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # Store gradients for each sample across checkpoints
    all_grads = {i: [] for i in range(len(dataloader.dataset))}

    print("Training base model to collect checkpoints for VoG...")
    for epoch in range(epochs):
        for batch_idx, batch in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")):

            # --- FIX: Unpack the tuple directly instead of using dictionary keys ---
            images, labels = batch[0].to(device), batch[1].to(device)

            images.requires_grad = True # Required to get gradients w.r.t input

            optimizer.zero_grad()
            outputs = model(images) # Pre-softmax activations

            # Gather pre-softmax activations for the true labels
            target_logits = outputs.gather(1, labels.view(-1, 1)).squeeze()

            # Compute gradients of pre-softmax w.r.t input pixels
            target_logits.backward(torch.ones_like(target_logits))

            # Store gradients. Taking absolute mean over color channels as per standard XAI practices.
            with torch.no_grad():
                pixel_grads = images.grad.abs().mean(dim=1) # Average over channels

                start_idx = batch_idx * dataloader.batch_size
                for i, grad in enumerate(pixel_grads):
                    all_grads[start_idx + i].append(grad.cpu().numpy())

            optimizer.step()

    # Calculate Variance
    vog_scores = np.zeros(len(dataloader.dataset))
    for i in range(len(dataloader.dataset)):
        # Stack gradients: shape (K, H, W)
        k_grads = np.stack(all_grads[i])
        # Variance across K checkpoints
        pixel_variance = np.var(k_grads, axis=0)
        # Mean across pixels
        vog_scores[i] = np.mean(pixel_variance)

    return vog_scores

# Generate VoG scores using a Frozen model to simulate early-stage dynamics
vog_model = get_model(frozen=False)
# Note: Dataloader shuffle must be False to track indices correctly
sequential_loader = DataLoader(train_subset, batch_size=32, shuffle=False)
vog_scores = compute_vog(vog_model, sequential_loader, epochs=3)

Training base model to collect checkpoints for VoG...


Epoch 3/3: 100%|██████████| 63/63 [00:24<00:00,  2.53it/s]


In [13]:
# Sort indices by VoG score (Ascending)
# Low VoG = Easier examples; High VoG = Harder examples
sorted_indices = np.argsort(vog_scores)

total_samples = len(sorted_indices)
split_idx = int(total_samples * 0.5) # Take 50% subsets

# Create subsets
low_vog_indices = sorted_indices[:split_idx]
high_vog_indices = sorted_indices[split_idx:]

low_vog_dataset = Subset(train_subset, low_vog_indices)
high_vog_dataset = Subset(train_subset, high_vog_indices)

loader_full = DataLoader(train_subset, batch_size=32, shuffle=True)
loader_low = DataLoader(low_vog_dataset, batch_size=32, shuffle=True)
loader_high = DataLoader(high_vog_dataset, batch_size=32, shuffle=True)

print(f"Total samples: {total_samples}")
print(f"Low VoG subset: {len(low_vog_dataset)} samples")
print(f"High VoG subset: {len(high_vog_dataset)} samples")

Total samples: 2000
Low VoG subset: 1000 samples
High VoG subset: 1000 samples


In [14]:
def train_and_evaluate(model, train_dl, val_dl, epochs=5):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for batch in train_dl:
            # --- FIX: Unpack the tuple directly ---
            images, labels = batch[0].to(device), batch[1].to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_dl:
            # --- FIX: Unpack the tuple directly ---
            images, labels = batch[0].to(device), batch[1].to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

In [15]:
print("--- Frozen Model (Linear Probe) ---")
acc_full_frozen = train_and_evaluate(get_model(frozen=True), loader_full, val_loader)
print(f"Accuracy on Full Dataset: {acc_full_frozen:.2f}%")

acc_low_frozen = train_and_evaluate(get_model(frozen=True), loader_low, val_loader)
print(f"Accuracy on Low VoG (Easy): {acc_low_frozen:.2f}%")

acc_high_frozen = train_and_evaluate(get_model(frozen=True), loader_high, val_loader)
print(f"Accuracy on High VoG (Hard): {acc_high_frozen:.2f}%")


print("\n--- Unfrozen Model (Fine-Tuning) ---")
# Using fewer epochs for unfrozen to save time in Colab
acc_full_unfrozen = train_and_evaluate(get_model(frozen=False), loader_full, val_loader, epochs=2)
print(f"Accuracy on Full Dataset: {acc_full_unfrozen:.2f}%")

acc_low_unfrozen = train_and_evaluate(get_model(frozen=False), loader_low, val_loader, epochs=2)
print(f"Accuracy on Low VoG (Easy): {acc_low_unfrozen:.2f}%")

acc_high_unfrozen = train_and_evaluate(get_model(frozen=False), loader_high, val_loader, epochs=2)
print(f"Accuracy on High VoG (Hard): {acc_high_unfrozen:.2f}%")

--- Frozen Model (Linear Probe) ---
Accuracy on Full Dataset: 28.87%
Accuracy on Low VoG (Easy): 28.92%
Accuracy on High VoG (Hard): 28.74%

--- Unfrozen Model (Fine-Tuning) ---
Accuracy on Full Dataset: 23.80%
Accuracy on Low VoG (Easy): 27.69%
Accuracy on High VoG (Hard): 25.25%


In [16]:
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

# --- Configuration ---
# Set to 90 for full paper replication, keeping it at 10 for Colab testing
TOTAL_EPOCHS = 10
CHECKPOINT_DIR = "./model_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 1. Initialize an unfrozen ResNet50
def get_fresh_model():
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 10)
    return model.to(device)

base_model = get_fresh_model()

# 2. Train and Save Checkpoints
def train_and_checkpoint(model, train_loader, epochs):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    print(f"Training for {epochs} epochs and saving checkpoints...")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images, labels = batch[0].to(device), batch[1].to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        # Save the model state at the end of each epoch
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch+1}.pt")
        torch.save(model.state_dict(), ckpt_path)
        print(f"Saved {ckpt_path} - Loss: {running_loss/len(train_loader):.4f}")

# Execute full training run
train_and_checkpoint(base_model, train_loader, TOTAL_EPOCHS)


# 3. Compute VoG from specific checkpoints
def compute_vog_from_checkpoints(model_arch, dataloader, checkpoint_paths):
    """Calculates VoG using pre-saved model checkpoints."""
    criterion = nn.CrossEntropyLoss()
    all_grads = {i: [] for i in range(len(dataloader.dataset))}

    print(f"Computing VoG across {len(checkpoint_paths)} checkpoints...")
    for ckpt_path in checkpoint_paths:
        # Load the specific checkpoint
        model_arch.load_state_dict(torch.load(ckpt_path))
        model_arch.train() # Must be in train mode to get gradients

        for batch_idx, batch in enumerate(dataloader):
            images, labels = batch[0].to(device), batch[1].to(device)
            images.requires_grad = True

            model_arch.zero_grad()
            outputs = model_arch(images)

            # Gather pre-softmax activations for the true labels
            target_logits = outputs.gather(1, labels.view(-1, 1)).squeeze()
            target_logits.backward(torch.ones_like(target_logits))

            with torch.no_grad():
                pixel_grads = images.grad.abs().mean(dim=1)
                start_idx = batch_idx * dataloader.batch_size
                for i, grad in enumerate(pixel_grads):
                    all_grads[start_idx + i].append(grad.cpu().numpy())

    # Calculate Variance
    vog_scores = np.zeros(len(dataloader.dataset))
    for i in range(len(dataloader.dataset)):
        k_grads = np.stack(all_grads[i])
        pixel_variance = np.var(k_grads, axis=0)
        vog_scores[i] = np.mean(pixel_variance)

    return vog_scores

# 4. Define Checkpoint Sets (Early vs Late)
# The paper uses the first 3 epochs for Early, and last 3 for Late
early_checkpoints = [os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{i}.pt") for i in [1, 2, 3]]
late_checkpoints = [os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{i}.pt") for i in [TOTAL_EPOCHS-2, TOTAL_EPOCHS-1, TOTAL_EPOCHS]]

# Note: Dataloader shuffle MUST be False to track indices correctly
sequential_loader = DataLoader(train_subset, batch_size=32, shuffle=False)

# Compute Early and Late VoG
print("\n--- Calculating Early VoG ---")
early_vog_scores = compute_vog_from_checkpoints(get_fresh_model(), sequential_loader, early_checkpoints)

print("\n--- Calculating Late VoG ---")
late_vog_scores = compute_vog_from_checkpoints(get_fresh_model(), sequential_loader, late_checkpoints)


# 5. Evaluate the Base Model's Error Rate on VoG Subsets
def evaluate_subset_error(model, full_dataset, indices, name):
    model.eval()
    subset = Subset(full_dataset, indices)
    loader = DataLoader(subset, batch_size=32, shuffle=False)

    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            images, labels = batch[0].to(device), batch[1].to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    error_rate = 100 - (100 * correct / total)
    print(f"{name} Error Rate: {error_rate:.2f}%")

# Sort data into High and Low percentiles (Top 25% and Bottom 25%)
total_samples = len(early_vog_scores)
top_idx = int(total_samples * 0.75)
bottom_idx = int(total_samples * 0.25)

early_sorted = np.argsort(early_vog_scores)
late_sorted = np.argsort(late_vog_scores)

print("\n=== FINAL RESULTS: THE VoG FLIP ===")
print("In Early Training, High VoG should have LOWER error (model learns them fast).")
print("In Late Training, High VoG should have HIGHER error (model struggles to memorize them).\n")

print("[Early Stage VoG Analysis]")
evaluate_subset_error(base_model, train_subset, early_sorted[:bottom_idx], "Lowest 25% Early VoG")
evaluate_subset_error(base_model, train_subset, early_sorted[top_idx:], "Highest 25% Early VoG")

print("\n[Late Stage VoG Analysis]")
evaluate_subset_error(base_model, train_subset, late_sorted[:bottom_idx], "Lowest 25% Late VoG")
evaluate_subset_error(base_model, train_subset, late_sorted[top_idx:], "Highest 25% Late VoG")

Training for 10 epochs and saving checkpoints...


Epoch 1/10: 100%|██████████| 63/63 [00:27<00:00,  2.32it/s]


Saved ./model_checkpoints/checkpoint_epoch_1.pt - Loss: 0.2493


Epoch 2/10: 100%|██████████| 63/63 [00:24<00:00,  2.56it/s]


Saved ./model_checkpoints/checkpoint_epoch_2.pt - Loss: 0.1014


...

Saved ./model_checkpoints/checkpoint_epoch_8.pt - Loss: 0.0150


Epoch 9/10: 100%|██████████| 63/63 [00:24<00:00,  2.54it/s]


Saved ./model_checkpoints/checkpoint_epoch_9.pt - Loss: 0.0265


Epoch 10/10: 100%|██████████| 63/63 [00:24<00:00,  2.54it/s]


Saved ./model_checkpoints/checkpoint_epoch_10.pt - Loss: 0.0629

--- Calculating Early VoG ---
Computing VoG across 3 checkpoints...

--- Calculating Late VoG ---
Computing VoG across 3 checkpoints...

=== FINAL RESULTS: THE VoG FLIP ===
In Early Training, High VoG should have LOWER error (model learns them fast).
In Late Training, High VoG should have HIGHER error (model struggles to memorize them).

[Early Stage VoG Analysis]
Lowest 25% Early VoG Error Rate: 2.00%
Highest 25% Early VoG Error Rate: 0.80%

[Late Stage VoG Analysis]
Lowest 25% Late VoG Error Rate: 2.00%
Highest 25% Late VoG Error Rate: 1.20%
